# 📊 Portafolio de Inversión para el Inversor Joven Conservador

**Curso:** Manejo de Datos  
**Enfoque:** Análisis cuantitativo de un portafolio de largo plazo (10–20 años)

---

## 🎯 Filosofía de inversión: ¿Por qué ser conservador siendo joven?

Existe una idea popular que dice *"eres joven, puedes asumir más riesgo"*. Eso es parcialmente cierto — tienes tiempo para recuperarte de caídas. Sin embargo, un **joven conservador** prioriza la consistencia sobre los rendimientos explosivos, por razones muy racionales:

1. **El interés compuesto favorece la paciencia.** Un rendimiento del 9% anual durante 20 años multiplica tu capital por **5.6x**. No necesitas apuestas agresivas.
2. **Las pérdidas grandes tardan más en recuperarse.** Una caída del 50% requiere un +100% para volver al punto de partida.
3. **La volatilidad afecta el comportamiento.** Los inversores que ven caer su portafolio un 40% suelen vender en el peor momento (pánico).

### 📐 Perfil del inversor que modelamos:
- 🎓 25–35 años, inicio de vida profesional
- 🎯 Horizonte de inversión: 15–20 años
- 💡 Objetivo: Crecimiento patrimonial con mínima intervención (estrategia *buy & hold*)
- 🛡️ Tolerancia al riesgo: Baja-Media — acepta volatilidad moderada, evita activos especulativos

---

## 🏗️ Construcción del Portafolio

### ¿Por qué ETFs y no acciones individuales?

Un **ETF (Exchange-Traded Fund)** es una canasta de activos que cotiza en bolsa como si fuera una sola acción. Para un inversor conservador son ideales porque:
- 📦 **Diversificación inmediata** → SPY contiene las 500 empresas más grandes de EE.UU.
- 💸 **Costos bajos** → comisiones de 0.03%–0.20% vs fondos activos que cobran 1%–2%
- 🔍 **Transparencia** → sabes exactamente qué tienes en todo momento
- 📈 **Track record probado** → décadas de datos históricos

### Los activos seleccionados y su justificación:

| Activo | Ticker | Peso | Clase | Justificación |
|--------|--------|------|-------|---------------|
| SPDR S&P 500 ETF | `SPY` | 40% | Renta Variable EE.UU. | Núcleo del portafolio. Exposición a las 500 empresas más grandes del mundo. Rendimiento histórico ~10% anual |
| Invesco QQQ (NASDAQ 100) | `QQQ` | 20% | Renta Variable Tecnología | Crecimiento en tecnología e innovación. Alto potencial a largo plazo con más volatilidad controlada |
| iShares MSCI World ETF | `URTH` | 15% | Renta Variable Global | Diversificación geográfica. Reduce dependencia de EE.UU. (Europa, Japón, mercados emergentes) |
| iShares Core US Aggregate Bond | `AGG` | 15% | Renta Fija | Bono diversificado de EE.UU. Ancla de estabilidad, bajo riesgo, correlación negativa con acciones |
| SPDR Gold Shares | `GLD` | 10% | Materia Prima | Oro como cobertura contra inflación y crisis. Se comporta bien cuando las acciones caen |

> **Nota metodológica:** Los pesos fueron definidos siguiendo el principio de la **Frontera Eficiente de Markowitz** — maximizar rendimiento esperado para un nivel de riesgo dado. Un portafolio 70% renta variable / 15% renta fija / 10% materias primas es clásico para perfil conservador-moderado.

---
## 🔧 Configuración e Instalación

In [1]:
# Instalar dependencias (solo primera vez)
# keras y tensorflow son necesarios para la sección de redes neuronales
!pip install yfinance plotly ipywidgets keras tensorflow scipy --quiet

# Activar widgets en Jupyter/Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Google Colab detectado — widgets activados")
except ImportError:
    print("✅ Jupyter local detectado")

print("✅ Instalación completa")

✅ Jupyter local detectado
✅ Instalación completa


In [2]:
# ─── Importaciones ────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta, date
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


---
## 📥 Sección 1: Descarga de Datos

Define el portafolio, selecciona el período y descarga los precios históricos desde Yahoo Finance.  
Presiona el botón para iniciar los datos quedan guardados en memoria para todas las secciones siguientes.

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL PORTAFOLIO (constantes globales)
# ════════════════════════════════════════════════════════════

# TODO 1: Define el diccionario PORTAFOLIO con los 5 activos.
# Cada activo debe tener: 'peso' (float), 'nombre' (str), 'color' (hex str), 'clase' (str)
# Activos: SPY (40%), QQQ (20%), URTH (15%), AGG (15%), GLD (10%)
PORTAFOLIO = {
    # TU CÓDIGO AQUÍ
}

TICKERS      = list(PORTAFOLIO.keys())
PESOS        = np.array([PORTAFOLIO[t]['peso'] for t in TICKERS])
DIAS_TRADING = 252    # Días hábiles bursátiles en un año
TASA_RF      = 0.045  # Tasa libre de riesgo (~T-Bills 2024)

# Constantes dadas — no modificar
DIAS_TRADING = 252    # Días hábiles bursátiles en un año
TASA_RF      = 0.045  # Tasa libre de riesgo (~T-Bills 2024)

# Variables globales — se llenarán al presionar el botón
PRECIOS         = None
RENDIMIENTOS    = None
REND_PORTAFOLIO = None

print('Portafolio configurado:')
for t, info in PORTAFOLIO.items():
    print(f'   {t}: {info["nombre"]} — {int(info["peso"]*100)}%')


📋 Portafolio configurado:
   SPY: S&P 500 ETF — 40%
   QQQ: NASDAQ 100 ETF — 20%
   URTH: MSCI World ETF — 15%
   AGG: US Bonds ETF — 15%
   GLD: Gold ETF — 10%


In [ ]:
# ─── Widgets de configuración (provistos — no modificar) ─────────────────────
start_date_picker = widgets.DatePicker(
    description='Fecha de Inicio:', value=date(2005, 1, 1),
    disabled=False, style={'description_width': 'initial'}
)
end_date_picker = widgets.DatePicker(
    description='Fecha de Fin:', value=date.today(),
    disabled=False, style={'description_width': 'initial'}
)
descarga_output = widgets.Output()


def descargar_datos(b=None):
    """
    Descarga precios históricos de Yahoo Finance, calcula rendimientos
    logarítmicos y el rendimiento ponderado del portafolio.
    Completa los TODO marcados con # TU CÓDIGO AQUÍ
    """
    global PRECIOS, RENDIMIENTOS, REND_PORTAFOLIO

    with descarga_output:
        descarga_output.clear_output()
        print(f'⏳ Descargando datos para: {TICKERS}')
        print(f'   Período: {start_date_picker.value} → {end_date_picker.value}\n')

        try:
            # TODO 3: Descarga los datos históricos con yfinance.
            # Usa: tickers=TICKERS, start, end desde los pickers, auto_adjust=True, progress=False
            raw = # TU CÓDIGO AQUÍ

            # TODO 4: Extrae solo los precios de cierre ('Close').
            # yfinance puede devolver MultiIndex — maneja ambos casos.
            # Aplica .ffill().dropna() al final.
            if isinstance(raw.columns, pd.MultiIndex):
                PRECIOS = # TU CÓDIGO AQUÍ
            else:
                PRECIOS = # TU CÓDIGO AQUÍ

            # Rendimientos logarítmicos diarios: log(P_t / P_{t-1})
            # Son preferidos en finanzas porque son aditivos en el tiempo
            RENDIMIENTOS =  np.log(PRECIOS / PRECIOS.shift(1)).dropna()

            # # Rendimiento del portafolio = suma ponderada de rendimientos individuales
            REND_PORTAFOLIO = (RENDIMIENTOS * PESOS).sum(axis=1)

            print(f'✅ Datos descargados correctamente:')
            print(f'   → {len(PRECIOS):,} días de trading')
            print(f'   → Período: {PRECIOS.index[0].date()} → {PRECIOS.index[-1].date()}')
            print(f'   → Activos: {list(PRECIOS.columns)}')

            # TODO 7: Construye la tabla de métricas (DataFrame con índice TICKERS).
            # Columnas requeridas: 'Nombre', 'Peso (%)', 'Rend. Anual (%)', 'Volatilidad (%)',
            #                      'Sharpe Ratio', 'Rend. Total (%)'
            # Pistas:
            #   Rend. Anual  = media diaria × 252 × 100
            #   Volatilidad  = std diaria × sqrt(252) × 100
            #   Sharpe       = (Rend_anual/100 - TASA_RF) / (Vol/100)
            #   Rend. Total  = (precio_final / precio_inicial - 1) × 100
            metricas = pd.DataFrame(index=TICKERS)
            # TU CÓDIGO AQUÍ

            # TODO 8: Calcula rend. anual, volatilidad y Sharpe del portafolio combinado
            # usando REND_PORTAFOLIO (mismas fórmulas que arriba)
            rend_pa  = # TU CÓDIGO AQUÍ
            vol_pa   = # TU CÓDIGO AQUÍ
            sharpe_p = # TU CÓDIGO AQUÍ

            print('\n📊 Métricas del portafolio (período completo):')
            display(metricas)
            print(f'\n🏆 Portafolio combinado:')
            print(f'   Rendimiento anual:  {rend_pa:.2f}%')
            print(f'   Volatilidad anual:  {vol_pa:.2f}%')
            print(f'   Sharpe Ratio:       {sharpe_p:.3f}')

            # TODO 9: Grafica el rendimiento acumulado base 100.
            # - Para cada activo: normaliza su precio dividiéndolo por el primero × 100
            # - Para el portafolio: usa (1 + REND_PORTAFOLIO).cumprod() × 100
            # - El portafolio debe ir en negro, línea discontinua (linestyle='--'), linewidth=2.5
            # - Agrega título, ejes, leyenda y grid
            valor_port = # TU CÓDIGO AQUÍ

            fig, ax = plt.subplots(figsize=(11, 4))
            # TU CÓDIGO AQUÍ

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f'❌ Error al descargar datos: {e}')


# Botón — no modificar
button_descarga = widgets.Button(
    description='📥 Descargar Datos del Portafolio',
    button_style='primary',
    layout=Layout(width='300px', height='38px')
)
button_descarga.on_click(descargar_datos)
display(HBox([start_date_picker, end_date_picker]))
display(button_descarga, descarga_output)


## Resultado esperado: 


---
## 📈 Sección 2: Dashboard Interactivo

Selecciona el período y tipo de visualización, luego presiona el botón para actualizar la gráfica.

In [5]:
# ════════════════════════════════════════════════════════════
# DASHBOARD — Selección de período y tipo de gráfica
# ════════════════════════════════════════════════════════════

# Widgets provistos — no modificar
PERIODOS = {
    '1 Semana': 7, '1 Mes': 30, '3 Meses': 90, '6 Meses': 180,
    '1 Año': 365, '3 Años': 365*3, '5 Años': 365*5, '10 Años': 365*10, 'Máximo': 365*20
}
selector_periodo = widgets.ToggleButtons(
    options=list(PERIODOS.keys()), value='5 Años',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=Layout(width='100%')
)
selector_tipo = widgets.RadioButtons(
    options=[
        ('Rendimiento acumulado (Base 100)', 'norm'),
        ('Rendimientos diarios (%)', 'rend'),
        ('Volatilidad móvil 30 días', 'vol'),
        ('Drawdown (caída desde máximo)', 'dd'),
    ],
    value='norm', style={'description_width': 'initial'}, layout=Layout(width='370px')
)
checks_activos = [
    widgets.Checkbox(
        value=True,
        description=f"{t} — {PORTAFOLIO[t]['nombre']} ({int(PORTAFOLIO[t]['peso']*100)}%)",
        style={'description_width': 'initial'}, layout=Layout(width='400px')
    ) for t in TICKERS
]
check_portafolio = widgets.Checkbox(
    value=True, description='📊 Mostrar portafolio ponderado',
    style={'description_width': 'initial'}, layout=Layout(width='300px')
)
dashboard_output = widgets.Output()


def actualizar_dashboard(b=None):
    with dashboard_output:
        dashboard_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        activos_sel  = [t for t, cb in zip(TICKERS, checks_activos) if cb.value]
        mostrar_port = check_portafolio.value
        dias_max     = PERIODOS[selector_periodo.value]
        tipo_graf    = selector_tipo.value

        if not activos_sel and not mostrar_port:
            print('⚠️ Selecciona al menos un activo.')
            return

        # TODO 10: Filtra los datos al período seleccionado.
        # fecha_corte = última fecha - timedelta(days=dias_max)
        # Filtra PRECIOS, RENDIMIENTOS y REND_PORTAFOLIO desde esa fecha
        fecha_corte = # TU CÓDIGO AQUÍ
        precios_f   = # TU CÓDIGO AQUÍ
        rend_f      = # TU CÓDIGO AQUÍ
        rend_port_f = # TU CÓDIGO AQUÍ

        fig, ax = plt.subplots(figsize=(12, 5))

        # TODO 11: Implementa los 4 tipos de gráfica según tipo_graf:
        #
        # 'norm' → Rendimiento acumulado base 100 (igual que Sección 1)
        #
        # 'rend' → Barras de rendimientos diarios (%) por activo
        #          Si mostrar_port: línea del portafolio encima
        #
        # 'vol'  → Volatilidad anualizada con ventana móvil de 30 días
        #          Fórmula: rolling(30).std() × sqrt(252) × 100
        #
        # 'dd'   → Drawdown: (precio - precio_máximo_hasta_hoy) / precio_máximo × 100
        #          Usa fill_between para rellenar el área bajo cero
        #          Pista: precio_máximo_hasta_hoy = s.cummax()
        if tipo_graf == 'norm':
            titulo, ylabel = f'Rendimiento Acumulado (Base 100) — {selector_periodo.value}', 'Valor indexado'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'rend':
            titulo, ylabel = f'Rendimientos Diarios (%) — {selector_periodo.value}', 'Rendimiento (%)'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'vol':
            titulo, ylabel = f'Volatilidad Anualizada 30 días (%) — {selector_periodo.value}', 'Volatilidad (%)'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'dd':
            titulo, ylabel = f'Drawdown desde Máximo (%) — {selector_periodo.value}', 'Drawdown (%)'
            # TU CÓDIGO AQUÍ

        ax.set_title(titulo, fontsize=13, fontweight='bold')
        ax.set_xlabel('Fecha')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # TODO 12: Muestra la tabla de métricas del período seleccionado.
        # Columnas: 'Nombre', 'Rend. Anual (%)', 'Volatilidad (%)', 'Sharpe', 'Rend. Total (%)'
        if activos_sel and not rend_f[activos_sel].empty:
            m = pd.DataFrame(index=activos_sel)
            # TU CÓDIGO AQUÍ
            print(f'\n📊 Métricas del período seleccionado ({selector_periodo.value}):')
            display(m)


# Layout — no modificar
button_dashboard = widgets.Button(
    description='🔄 Actualizar Gráfica', button_style='info',
    layout=Layout(width='200px', height='38px')
)
button_dashboard.on_click(actualizar_dashboard)
panel_checks = VBox(
    [widgets.HTML('<b>📦 Activos</b>')] + checks_activos + [check_portafolio],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='430px')
)
panel_tipo = VBox(
    [widgets.HTML('<b>📉 Visualización</b>'), selector_tipo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='380px')
)
panel_periodo = VBox(
    [widgets.HTML('<b>📅 Período</b>'), selector_periodo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px')
)
display(HBox([panel_checks, panel_tipo], layout=Layout(gap='12px', margin='0 0 10px 0')))
display(panel_periodo)
display(button_dashboard)
display(dashboard_output)


Button(button_style='info', description='🔄 Actualizar Gráfica', layout=Layout(height='38px', width='200px'), s…

Output()

---
## 📐 Sección 3: Análisis Cuantitativo — Correlaciones y Simulación histórica

In [6]:
# ─── Análisis cuantitativo: correlación + simulación histórica ────────────────

analisis_output = widgets.Output()


def analisis_cuantitativo(b=None):
    with analisis_output:
        analisis_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        # TODO 13: Calcula la matriz de correlación entre los rendimientos.
        # Usa .corr().round(3) sobre RENDIMIENTOS
        corr = # TU CÓDIGO AQUÍ
        nombres = [PORTAFOLIO[t]['nombre'] for t in TICKERS]

        # TODO 14: Grafica la matriz de correlación como heatmap.
        # - Usa ax.imshow con cmap='RdYlGn', vmin=-1, vmax=1
        # - Agrega colorbar, etiquetas en ejes x e y (usa nombres)
        # - Muestra el valor numérico en cada celda con ax.text()
        fig, ax = plt.subplots(figsize=(7, 5))
        # TU CÓDIGO AQUÍ

        ax.set_title('Correlación entre Rendimientos del Portafolio', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print('💡 El oro (GLD) y los bonos (AGG) tienen correlación baja/negativa con acciones → buena diversificación.')

        # TODO 15: Simulación histórica — ¿cuánto valdría $10,000 invertidos hace 10 años?
        # 1. Filtra precios y rendimiento del portafolio a los últimos 10 años
        # 2. Calcula el valor acumulado: INVERSION × (1 + rend_port_10y).cumprod()
        # 3. Grafica el valor de cada activo individual (100% en ese activo) y el portafolio
        # 4. Imprime: inversión inicial, valor final, ganancia y múltiplo
        INVERSION = 10_000
        # TU CÓDIGO AQUÍ


button_analisis = widgets.Button(
    description='📊 Mostrar Correlaciones y Simulación Histórica',
    button_style='success', layout=Layout(width='380px', height='38px')
)
button_analisis.on_click(analisis_cuantitativo)
display(button_analisis, analisis_output)


Button(button_style='success', description='📊 Mostrar Correlaciones y Simulación Histórica', layout=Layout(hei…

Output()

---
## 🎲 Sección 4: Proyección Monte Carlo — SPY (S&P 500)

### ¿Qué es Monte Carlo?

La simulación de **Monte Carlo** es una técnica matemática que usa **números aleatorios** para modelar situaciones con incertidumbre. En finanzas la usamos para responder: *¿cómo podría evolucionar el precio de un activo en el futuro?*

**Idea central:** Si sabemos cuál ha sido el rendimiento promedio y la volatilidad histórica de un activo, podemos simular miles de posibles "futuros" respetando esas estadísticas. Al ver el conjunto de todos esos futuros, obtenemos una distribución de probabilidad de los precios.

### El modelo: Movimiento Browniano Geométrico (GBM)

$$S_{t+1} = S_t \cdot \exp\left[(\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} \cdot Z\right]$$

Donde:
- $\mu$ = rendimiento promedio diario (estimado históricamente)
- $\sigma$ = volatilidad diaria (estimada históricamente)
- $Z \sim \mathcal{N}(0,1)$ = número aleatorio normal estándar
- El término $-\sigma^2/2$ es la **corrección de Jensen** que evita sesgo al tomar logaritmos

Usa los sliders para configurar la simulación y presiona el botón para ejecutarla.

In [7]:
# ════════════════════════════════════════════════════════════
# MONTE CARLO — SPY (activo único)
# ════════════════════════════════════════════════════════════

ACTIVO_MC = 'SPY'

# Widgets provistos — no modificar
N_slider_mc = widgets.IntSlider(
    value=500, min=100, max=2000, step=100,
    description='N simulaciones:', style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
horizonte_slider_mc = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description='Horizonte (años):', style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
mc_output = widgets.Output()


def simular_montecarlo_spy(b=None):
    with mc_output:
        mc_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        np.random.seed(42)
        N       = N_slider_mc.value
        H_ANIOS = horizonte_slider_mc.value
        N_DIAS  = H_ANIOS * DIAS_TRADING

        # TODO 16: Estima los parámetros del modelo GBM con los últimos 5 años de SPY.
        # mu_d    = rendimiento promedio diario
        # sigma_d = volatilidad diaria (std)
        # S0      = precio actual (último precio disponible)
        ultimos_5y = PRECIOS.index[-1] - timedelta(days=5*365)
        rend_spy   = RENDIMIENTOS.loc[RENDIMIENTOS.index >= ultimos_5y, ACTIVO_MC]
        mu_d    = # TU CÓDIGO AQUÍ
        sigma_d = # TU CÓDIGO AQUÍ
        S0      = # TU CÓDIGO AQUÍ

        print(f'📊 Parámetros estimados para {ACTIVO_MC} (últimos 5 años):')
        print(f'   Precio actual:             ${S0:.2f}')
        print(f'   Rendimiento diario (µ):    {mu_d:.4%}')
        print(f'   Volatilidad diaria (σ):    {sigma_d:.4%}')
        print(f'   Rendimiento anual (µ·252): {mu_d*252:.2%}')
        print(f'   Volatilidad anual (σ·√252):{sigma_d*np.sqrt(252):.2%}')
        print(f'\n🎲 Simulando {N:,} escenarios para {H_ANIOS} años...')

        # TODO 17: Implementa la simulación GBM vectorizada.
        # Fórmula: S_{t+1} = S_t * exp(drift + shock)
        # donde:
        #   drift  = mu_d - 0.5 * sigma_d**2   (corrección de Jensen)
        #   shocks = sigma_d * Z               (Z ~ N(0,1), shape: N_DIAS × N)
        # Usa np.cumsum sobre el eje 0 y luego np.exp
        # Al final añade S0 como fila inicial con np.vstack
        Z = np.random.standard_normal((N_DIAS, N))
        # TU CÓDIGO AQUÍ
        # precios_sim debe tener shape (N_DIAS+1, N)

        # TODO 18: Calcula los percentiles p5, p25, p50, p75, p95 por día.
        # Usa np.percentile(..., axis=1)
        # TU CÓDIGO AQUÍ

        dias = np.arange(N_DIAS + 1)

        # TODO 19: Gráfica 1 — trayectorias + bandas de confianza.
        # - Dibuja máximo 200 trayectorias individuales (color steelblue, alpha bajo)
        # - Banda 90%: fill_between(p5, p95)
        # - Banda 50%: fill_between(p25, p75)
        # - Líneas de p95, p50 (mediana) y p5 con etiquetas del precio final
        # - Punto de inicio (S0 en día 0)
        fig, ax = plt.subplots(figsize=(12, 5))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 20: Gráfica 2 — histograma de precios finales.
        # - Histograma de precios_sim[-1, :] con bins=60, density=True
        # - Líneas verticales para P5, Mediana, P95 y precio actual (S0)
        precios_finales = precios_sim[-1, :]
        fig2, ax2 = plt.subplots(figsize=(9, 4))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 21: Calcula e imprime las probabilidades:
        # - Prob. de terminar con más dinero que S0
        # - Prob. de duplicar (> S0 * 2)
        # - Prob. de perder más del 50% (< S0 * 0.5)
        # TU CÓDIGO AQUÍ


button_mc = widgets.Button(
    description='🎲 Ejecutar Monte Carlo (SPY)', button_style='warning',
    layout=Layout(width='280px', height='38px')
)
button_mc.on_click(simular_montecarlo_spy)
display(VBox([N_slider_mc, horizonte_slider_mc, button_mc, mc_output]))


---
## 🤖 Sección 5: Comparativa — Monte Carlo vs Red Neuronal LSTM

### ¿Por qué comparar dos métodos?

| Característica | Monte Carlo (GBM) | Red Neuronal LSTM |
|----------------|-------------------|-------------------|
| **Tipo de modelo** | Estadístico / Probabilístico | Machine Learning / Determinístico |
| **Supuesto clave** | Rendimientos son ruido normal | El mercado tiene patrones aprendibles |
| **Salida** | Distribución (miles de escenarios) | Una sola trayectoria |
| **Interpretabilidad** | Alta — µ y σ tienen significado claro | Baja — "caja negra" |
| **Captura tendencias** | No | Sí (parcialmente) |

### ¿Qué es una LSTM?

Una **LSTM (Long Short-Term Memory)** es un tipo de red neuronal diseñada para series de tiempo. Tiene "memoria" — puede recordar patrones de hace varios pasos.

> ⚠️ Las LSTM son buenas para ajustar datos pasados, pero **no son bolas de cristal**. A largo plazo acumulan errores porque los mercados tienen aleatoriedad irreducible.

Presiona el botón para entrenar la LSTM y hacer la comparativa. **Puede tardar 1–3 minutos.**

In [8]:
# ════════════════════════════════════════════════════════════
# COMPARATIVA: MONTE CARLO vs LSTM
# ════════════════════════════════════════════════════════════
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import keras
from keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

ACTIVO_LSTM = 'SPY'
VENTANA     = 60

modelo_lstm          = None
predicciones_futuras = None
scaler_lstm          = None

comparativa_output = widgets.Output()


def entrenar_y_comparar(b=None):
    global modelo_lstm, predicciones_futuras, scaler_lstm

    with comparativa_output:
        comparativa_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        print(f'⏳ Entrenando LSTM para {ACTIVO_LSTM}... (1–3 minutos en CPU)')
        precios_spy = PRECIOS[[ACTIVO_LSTM]].copy()

        # TODO 22: Normaliza los precios al rango [0,1] usando MinMaxScaler.
        # Guarda el scaler en scaler_lstm (lo necesitas después para invertir la transformación)
        scaler_lstm = # TU CÓDIGO AQUÍ
        precios_esc = # TU CÓDIGO AQUÍ

        # TODO 23: Divide en 80% train / 20% test (sin mezclar — respeta el orden temporal).
        split      = # TU CÓDIGO AQUÍ
        train_data = # TU CÓDIGO AQUÍ
        test_data  = # TU CÓDIGO AQUÍ

        # TODO 24: Implementa la función crear_seq(datos, ventana).
        # Genera pares (X, y) donde X = últimos `ventana` precios, y = precio siguiente.
        def crear_seq(datos, ventana):
            X, y = [], []
            # TU CÓDIGO AQUÍ
            return np.array(X), np.array(y)

        X_train, y_train = crear_seq(train_data, VENTANA)
        X_test, y_test   = crear_seq(np.concatenate([train_data[-VENTANA:], test_data]), VENTANA)

        # TODO 25: Reshapea X_train y X_test a (muestras, pasos_tiempo, 1)
        # La LSTM espera 3 dimensiones
        X_train = # TU CÓDIGO AQUÍ
        X_test  = # TU CÓDIGO AQUÍ

        # TODO 26: Construye el modelo LSTM con esta arquitectura:
        # LSTM(64, return_sequences=True) → Dropout(0.20)
        # → LSTM(32, return_sequences=False) → Dropout(0.20)
        # → Dense(16, activation='relu') → Dense(1)
        # Compila con optimizer='adam', loss='mean_squared_error'
        keras.utils.set_random_seed(42)
        modelo_lstm = models.Sequential([
            # TU CÓDIGO AQUÍ
        ], name='LSTM_SPY')
        # TU CÓDIGO AQUÍ (compile)
        modelo_lstm.summary()

        # TODO 27: Entrena el modelo.
        # epochs=30, batch_size=32, validation_split=0.10, verbose=0
        hist = # TU CÓDIGO AQUÍ

        print(f'\n✅ Entrenamiento completo')
        print(f'   Loss final (train): {hist.history["loss"][-1]:.6f}')
        print(f'   Loss final (val):   {hist.history["val_loss"][-1]:.6f}')

        # TODO 28: Grafica la curva de aprendizaje (train loss vs val loss por época)
        fig0, ax0 = plt.subplots(figsize=(8, 3))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 29: Evalúa el modelo en el conjunto de prueba.
        # 1. Predice con modelo_lstm.predict(X_test)
        # 2. Desnormaliza con scaler_lstm.inverse_transform()
        # 3. Calcula RMSE, MAE y MAPE
        # TU CÓDIGO AQUÍ

        # TODO 30: Grafica Precio Real vs Predicción LSTM en el período de prueba.
        fechas_prueba = precios_spy.index[split:]
        fig1, ax1 = plt.subplots(figsize=(12, 5))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 31: Proyecta 3 años (252×3 días) hacia el futuro con ventana deslizante.
        # En cada paso: predice el siguiente precio, agrégalo a la secuencia, repite.
        # Guarda en predicciones_futuras (desnormalizado)
        DIAS_FUTURO = 252 * 3
        secuencia = precios_esc[-VENTANA:].reshape(1, VENTANA, 1).copy()
        pred_futuras_esc = []
        # TU CÓDIGO AQUÍ
        predicciones_futuras = scaler_lstm.inverse_transform(
            np.array(pred_futuras_esc).reshape(-1, 1)
        ).flatten()

        # TODO 32: Gráfica comparativa Monte Carlo vs LSTM (3 años).
        # - Re-ejecuta Monte Carlo con N=500 escenarios y 3 años
        # - Muestra el histórico del último año a la izquierda del eje 0
        # - MC: bandas de confianza + mediana (color steelblue)
        # - LSTM: línea única (color tomato)
        # - Línea vertical en x=0 ('Hoy')
        np.random.seed(42)
        S0_lstm = float(precios_spy[ACTIVO_LSTM].iloc[-1])
        fig2, ax2 = plt.subplots(figsize=(13, 6))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()


button_lstm = widgets.Button(
    description='🤖 Entrenar LSTM y Comparar con Monte Carlo',
    button_style='danger', layout=Layout(width='380px', height='38px')
)
button_lstm.on_click(entrenar_y_comparar)
display(button_lstm, comparativa_output)


E0000 00:00:1777594879.251020     545 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777594879.256892     545 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777594879.271518     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271533     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271535     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271536     545 computation_placer.cc:177] computation placer already registered. Please check linka

Button(button_style='danger', description='🤖 Entrenar LSTM y Comparar con Monte Carlo', layout=Layout(height='…

Output()

---
## 💼 Sección 6: Proyección Monte Carlo — Portafolio Completo

### ¿Por qué proyectar el portafolio y no solo SPY?

Cuando proyectamos los 5 activos **juntos**, capturamos la **correlación entre ellos**. Si SPY cae, AGG (bonos) suele subir — ese efecto reduce el riesgo total más que la suma de los riesgos individuales.

Para respetar las correlaciones usamos la **Descomposición de Cholesky**: factoriza la matriz de covarianza $\Sigma = L \cdot L^T$ para generar rendimientos aleatorios correlacionados.

$$\mathbf{r}_{correlacionado} = L \cdot \mathbf{Z}, \quad \mathbf{Z} \sim \mathcal{N}(0, I)$$

Configura los parámetros y presiona el botón.

In [10]:
# ════════════════════════════════════════════════════════════
# MONTE CARLO — PORTAFOLIO COMPLETO (5 activos correlacionados)
# ════════════════════════════════════════════════════════════

# Widgets provistos — no modificar
N_slider_port = widgets.IntSlider(
    value=1000, min=100, max=3000, step=100,
    description='N simulaciones:', style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
horizonte_slider_port = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description='Horizonte (años):', style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
inversion_input = widgets.BoundedIntText(
    value=10000, min=100, max=1_000_000, step=500,
    description='Inversión inicial (USD):', style={'description_width': 'initial'},
    layout=Layout(width='350px')
)
portafolio_mc_output = widgets.Output()


def simular_portafolio_mc(b=None):
    with portafolio_mc_output:
        portafolio_mc_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        np.random.seed(42)
        N       = N_slider_port.value
        H_ANIOS = horizonte_slider_port.value
        N_DIAS  = H_ANIOS * DIAS_TRADING
        INV     = inversion_input.value

        print(f'⏳ Simulando portafolio completo: {N:,} escenarios × {H_ANIOS} años')

        # TODO 33: Estima los parámetros con los últimos 5 años.
        # medias   = vector de medias diarias de cada activo (shape 5,)
        # cov_d    = matriz de covarianza diaria (shape 5×5)
        # varianzas = diagonal de cov_d (shape 5,)
        ultimos_5y = PRECIOS.index[-1] - timedelta(days=5*365)
        rend_rec   = RENDIMIENTOS.loc[RENDIMIENTOS.index >= ultimos_5y]
        medias    = # TU CÓDIGO AQUÍ
        cov_d     = # TU CÓDIGO AQUÍ
        varianzas = # TU CÓDIGO AQUÍ

        # TODO 34: Aplica la descomposición de Cholesky a cov_d.
        # L = np.linalg.cholesky(cov_d)
        # Esto permite generar ruido correlacionado: shocks = L @ Z
        L = # TU CÓDIGO AQUÍ

        # TODO 35: Implementa el bucle de simulación para N escenarios.
        # Para cada escenario:
        #   1. Genera Z ~ N(0,1) de shape (5, N_DIAS)
        #   2. shocks = L @ Z  → ruido correlacionado
        #   3. drift_vec = medias - 0.5 * varianzas  (corrección de Jensen)
        #   4. rend_activos = drift_vec.reshape(-1,1) + shocks
        #   5. rend_port_d  = PESOS @ rend_activos  (rendimiento diario del portafolio)
        #   6. valores_port[1:, sim] = INV * np.cumprod(np.exp(rend_port_d))
        valores_port = np.zeros((N_DIAS + 1, N))
        valores_port[0, :] = INV
        drift_vec = medias - 0.5 * varianzas
        # TU CÓDIGO AQUÍ

        # TODO 36: Calcula percentiles p5, p25, p50, p75, p95 por día.
        # TU CÓDIGO AQUÍ

        dias_p = np.arange(N_DIAS + 1)

        # TODO 37: Gráfica 1 — trayectorias + bandas (igual estructura que Sección 4).
        # Incluye todas las líneas de percentiles y la línea horizontal de inversión inicial.
        fig, ax = plt.subplots(figsize=(12, 6))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 38: Gráfica 2 — distribución de valores finales.
        vals_finales = valores_port[-1, :]
        fig2, ax2 = plt.subplots(figsize=(9, 4))
        # TU CÓDIGO AQUÍ
        plt.tight_layout()
        plt.show()

        # TODO 39: Calcula e imprime las probabilidades y la tabla de percentiles finales.
        # Probabilidades: ganar, duplicar (2x), triplicar (3x), perder >25%
        # TU CÓDIGO AQUÍ


button_port_mc = widgets.Button(
    description='💼 Simular Portafolio Completo', button_style='primary',
    layout=Layout(width='300px', height='38px')
)
button_port_mc.on_click(simular_portafolio_mc)
display(VBox([N_slider_port, horizonte_slider_port, inversion_input,
              button_port_mc, portafolio_mc_output]))


---
## ✅ Conclusiones

**¿Por qué este portafolio es adecuado para un joven conservador?**

1. **Diversificación real:** 5 activos con correlaciones bajas entre sí — cuando las acciones caen, los bonos y el oro suelen estabilizarse.
2. **Exposición global:** A través de URTH, el portafolio no depende únicamente del mercado americano.
3. **Costos mínimos:** Todos los ETFs tienen expense ratios menores al 0.20% anual.

---

## 🔬 Conclusiones Metodológicas

### Monte Carlo (recomendado para planificación financiera):
- Genera una **distribución completa de escenarios** con probabilidades asociadas
- Es honesto sobre la incertidumbre — la banda se ensancha con el tiempo
- Ideal para: planificación de largo plazo, análisis de riesgo, decisiones de ahorro

### LSTM (complemento para análisis de corto plazo):
- Captura **tendencias y patrones** recientes en los datos
- Produce una sola trayectoria — no modela incertidumbre explícitamente
- Acumula errores en proyecciones largas
- Ideal para: señales de trading de corto plazo, análisis de momentum

> *Ningún modelo puede predecir el futuro con certeza. Monte Carlo lo admite abiertamente mostrando un rango de posibilidades. Para un inversor de largo plazo, esa honestidad sobre la incertidumbre es más valiosa que la aparente precisión de la red neuronal.*

---
> ⚠️ **Disclaimer:** Este notebook es exclusivamente educativo. No constituye asesoramiento financiero ni de inversión. Los rendimientos pasados no garantizan resultados futuros.